# Custom middleware  自定义中间件

## Hooks

### Node-style hooks  Node
按顺序在特定执行点运行。用于日志记录、验证和状态更新。

**Available hooks:**
- `before_agent` - 在代理启动之前（每次调用一次）
- `before_model` - 每次模型调用之前
- `after_model` - 每次模型响应之后
- `after_agent` - 代理完成后（每次调用一次）

In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

base_url = os.getenv("QWEN_BASE_URL")
api_key = os.getenv("QWEN_API_KEY")

model = ChatOpenAI(
    model="qwen3.5-plus-2026-02-15",
    base_url=base_url,
    api_key=api_key,
)

In [ ]:
from langchain.agents.middleware import before_model, after_model, AgentState
from langchain.messages import AIMessage, HumanMessage
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from typing import Any


@before_model(can_jump_to=["end"])
def check_message_limit(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    if len(state["messages"]) >= 50:
        return {
            "messages": [AIMessage("Conversation limit reached.")],
            "jump_to": "end"
        }
    return None

@after_model
def log_response(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"Model returned: {state['messages'][-1].content}")
    return None

agent = create_agent(
    model,
    middleware=[check_message_limit, log_response]
)
res = agent.invoke({
    "messages": [HumanMessage(content="Hello, how are you?")]
})
res

### Wrap-style hooks
拦截执行过程，并在调用处理程序时进行控制。可用于重试、缓存和转换。

可以决定处理程序是被调用零次（短路）、一次（正常流程）还是多次（重试逻辑）。
**Available hooks:** 
- `wrap_model_call` - 围绕每个模型调用
- `wrap_tool_call` - 围绕每次工具调用

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable


@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request)
        except Exception as e:
            if attempt == 2:
                raise
            print(f"Retry {attempt + 1}/3 after error: {e}")

agent = create_agent(
    model,
    middleware=[retry_model]
)
res = agent.invoke({
    "messages": [HumanMessage(content="Hello, how are you?")]
})
res

## State updates 
节点式和包装式钩子都可以更新代理状态。它们的机制有所不同：
- **Node-style hooks:** 直接返回一个字典。该字典会使用图的 reducer 应用于 agent 状态。
- **Wrap-style hooks:** 对于模型调用，返回一个包含 `Command` 的 `ExtendedModelResponse` ，以便在模型响应中注入状态更新。对于工具调用，直接返回一个 ` Command `。

In [ ]:
from langchain.agents.middleware import after_model, AgentState
from langgraph.runtime import Runtime
from typing import Any
from typing_extensions import NotRequired


class TrackingState(AgentState):
    model_call_count: NotRequired[int]


@after_model(state_schema=TrackingState)
def increment_after_model(state: TrackingState, runtime: Runtime) -> dict[str, Any] | None:
    return {"model_call_count": state.get("model_call_count", 0) + 1}

agent = create_agent(
    model,
    middleware=[increment_after_model]
)

res = agent.invoke({
    "messages": [HumanMessage(content="Hello, how are you?")]
})
res

In [ ]:
from typing import Callable
from langchain.agents.middleware import (
    wrap_model_call,
    ModelRequest,
    ModelResponse,
    AgentState,
    ExtendedModelResponse
)
from langgraph.types import Command
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from typing_extensions import NotRequired

class UsageTrackingState(AgentState):
    """Agent state with token usage tracking."""

    last_model_call_tokens: NotRequired[int]


@wrap_model_call(state_schema=UsageTrackingState)
def track_usage(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ExtendedModelResponse:
    response = handler(request)
    print("模型被叫了")
    return ExtendedModelResponse(
        model_response=response,
        command=Command(update={"last_model_call_tokens": 150}),
    )

agent = create_agent(
    model,
    checkpointer=InMemorySaver(),
    middleware=[track_usage]
)
config = {"configurable": {
    "thread_id": 1
}}
res = agent.invoke({
    "messages": [HumanMessage(content="Hello, how are you?")]
},
    config=config
)
res

### Composition with multiple middleware
当多个中间件层返回 ExtendedModelResponse 时，它​​们的命令组成如下：
- **命令通过 reducer 执行**： 每个 Command 都会成为一次独立的状态更新。对于消息而言，这意味着它们是累加的。
- **冲突时，外层优先**： 对于非 reducer 状态字段，命令先从内层应用，再从外层应用。最外层中间件的值在键冲突时优先。
- **可重试**： 如果外部中间件实现了可能导致多次调用 handler() 的逻辑（例如，重试逻辑），则会丢弃先前调用的命令。

In [ ]:
from typing import Annotated, Callable

from langchain.agents.middleware import (
    AgentMiddleware,
    AgentState,
    ExtendedModelResponse,
    ModelRequest,
    ModelResponse,
)
from langchain.messages import SystemMessage
from langgraph.types import Command
from typing_extensions import NotRequired


def _last_wins(_a: str, b: str) -> str:
    """Reducer: last writer wins (outer overwrites inner)."""
    return b


class CustomMiddlewareState(AgentState):
    """Agent state: trace_layer uses last-wins (outer wins), messages use additive reducer."""

    # Non-reducer field with last-wins: both middleware write; outermost value wins
    trace_layer: NotRequired[Annotated[str, _last_wins]]


class OuterMiddleware(AgentMiddleware):
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ExtendedModelResponse:
        response = handler(request)
        print("Outer middleware ran")
        return ExtendedModelResponse(
            model_response=response,
            command=Command(update={
                "trace_layer": "outer",
                "messages": [SystemMessage(content="[Outer ran]")],
            }),
        )


class InnerMiddleware(AgentMiddleware):
    """Adds trace_layer and message. Outer adds to same keys; trace_layer: outer wins, messages: additive."""

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ):
        response = handler(request)
        print("Inner middleware ran")
        return ExtendedModelResponse(
            model_response=response,
            command=Command(update={
                "trace_layer": "inner",
                "messages": [SystemMessage(content="[Inner ran]")],
            }),
        )

agent = create_agent(
    model,
    checkpointer=InMemorySaver(),
    middleware=[OuterMiddleware(), InnerMiddleware()]
)
res = agent.invoke({
    "messages": [HumanMessage(content="Hello, how are you?")]
},
    config={"configurable": {"thread_id": 1}}
)
res

## Create middleware
### Decorator-based middleware
适用于单钩子中间件，快速简便。使用装饰器封装各个函数。

**Node-style**:
@before_agent,@before_model,@after_model,@after_agent
**Wrap-style:**
@wrap_model_call,@wrap_tool_call 
**Convenience:**
@dynamic_prompt - 生成动态系统提示。

**When to use decorators:**
- 需要一个钩子
- 无需复杂配置
- 快速原型制作
### Class-based middleware 
对于具有多个钩子或配置的复杂中间件，类的功能更强大。当需要为同一个钩子定义同步和异步实现，或者想要在单个中间件中组合多个钩子时，请使用类。

**When to use classes:**
- 为同一个钩子定义同步和异步实现
- 单个中间件中需要多个钩子
- 需要复杂的配置（例如，可配置阈值、自定义模型）
- 跨项目复用，并进行初始化配置

## Custom state schema  
中间件可以使用自定义属性扩展代理的状态。这使得中间件能够：
- 跟踪执行过程中的状态 ：维护在代理的整个执行生命周期中保持不变的计数器、标志或其他值
- 在钩子之间共享数据 ：将信息从 before_model 传递到 after_model ，或在不同的中间件实例之间传递。
- 实现横切关注点 ：在不修改核心代理逻辑​​的情况下，添加诸如速率限制、使用情况跟踪、用户上下文或审计日志记录等功能。
- 做出条件决策 ：利用累积状态来确定是继续执行、跳转到不同的节点，还是动态地修改行为。

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.agents.middleware import AgentState, before_model, after_model
from typing_extensions import NotRequired
from typing import Any
from langgraph.runtime import Runtime


class CustomState(AgentState):
    model_call_count: NotRequired[int]
    user_id: NotRequired[str]


@before_model(state_schema=CustomState, can_jump_to=["end"])
def check_call_limit(state: CustomState, runtime: Runtime) -> dict[str, Any] | None:
    count = state.get("model_call_count", 0)
    if count > 10:
        return {"jump_to": "end"}
    return None


@after_model(state_schema=CustomState)
def increment_counter(state: CustomState, runtime: Runtime) -> dict[str, Any] | None:
    return {"model_call_count": state.get("model_call_count", 0) + 1}


agent = create_agent(
    model,
    middleware=[check_call_limit, increment_counter],
    tools=[],
)

# Invoke with custom state
result = agent.invoke({
    "messages": [HumanMessage("我是王小明")],
    "model_call_count": 11,
    "user_id": "user-123",
})
result

## Execution order 
**Key rules:**
- `before_* hooks:` 从第一个到最后一个
- `after_* hooks:` 从后到前（反向）
- `wrap_* hooks:` 嵌套式（第一个中间件包裹所有其他中间件）  

## Agent jumps
要提前退出中间件，请返回一个包含 jump_to 字典：

**可跳跃目标**：
- `end`: 跳转到代理执行的末尾（或第一个 after_agent 钩子）
- `tools`: 跳转到工具节点
- `model`: 跳转到模型节点（或第一个 before_model 钩子）

In [ ]:
from langchain.agents.middleware import after_model, hook_config, AgentState
from langchain.messages import AIMessage
from langgraph.runtime import Runtime
from typing import Any


@before_model
@hook_config(can_jump_to=["end"])
def check_for_blocked(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    last_message = state["messages"][-1]
    print(last_message)
    if "BLOCKED" in last_message.content:
        return {
            "messages": [AIMessage("I cannot respond to that request.")],
            "jump_to": "end"
        }
    return None

agent = create_agent(
    model,
    middleware=[check_for_blocked]
)
res = agent.invoke({
    "messages": [HumanMessage(content="Hello, how are you?"),SystemMessage(content="This message contains BLOCKED and should trigger the middleware.")]
})
res

## Best practices
1. 保持中间件的专注——每个中间件都应该做好一件事。
2. 优雅地处理错误——不要让中间件错误导致代理崩溃。
3. **使用合适的钩子类型 ：**
   - 用于顺序逻辑（日志记录、验证）的节点式编程
   - 控制流（重试、回退、缓存）的包装式
4. 清楚地记录所有自定义状态属性
5. 在集成之前，对中间件进行独立的单元测试。
6. 考虑执行顺序——将关键中间件放在列表的最前面
7. 尽可能使用内置中间件
## Dynamically selecting tools
- 更简短的提示 ——仅显示相关工具，降低复杂性
- 更准确的选择 ——模型能从更少的选项中做出正确的选择。
- 权限控制 - 根据用户访问权限动态筛选工具